# Project2 - GPT 2.0 Model

1. 기본 import

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

torch.manual_seed(42)
random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


2. .txt 파일 업로드

In [4]:
try:
    from google.colab import files

    uploaded = files.upload()
    filename = list(uploaded.keys())[0]

    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()

except:
    filename = "input.txt"

    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()

print("Loaded file:", filename)
print("Text length:", len(text))
print()
print(text[:500])

FileNotFoundError: [Errno 2] No such file or directory: 'input.txt'

3. 문자 vocabulary 만들기

In [6]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return "".join([itos[i] for i in ids])

data = torch.tensor(encode(text), dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Characters:")
print("".join(chars))
print()
print("Encoded sample:", data[:20])
print("Decoded sample:", decode(data[:20].tolist()))

NameError: name 'text' is not defined

4. Hyperparameter 설정

In [ ]:
batch_size = 64
block_size = 128
max_iters = 3000
eval_interval = 300
learning_rate = 3e-4
eval_iters = 100

n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2

assert n_embd % n_head == 0

print("batch_size:", batch_size)
print("block_size:", block_size)
print("n_embd:", n_embd)
print("n_head:", n_head)
print("n_layer:", n_layer)

5. Train / Validation 데이터 나누기

In [ ]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Train length:", len(train_data))
print("Validation length:", len(val_data))

if len(train_data) <= block_size or len(val_data) <= block_size:
    raise ValueError("텍스트가 너무 짧습니다. 더 긴 영어 텍스트를 사용하거나 block_size를 줄이세요.")

6. get_batch

In [ ]:
def get_batch(split):
    source = train_data if split == "train" else val_data

    ix = torch.randint(len(source) - block_size, (batch_size,))

    x = torch.stack([
        source[i : i + block_size]
        for i in ix
    ])

    y = torch.stack([
        source[i + 1 : i + block_size + 1]
        for i in ix
    ])

    x = x.to(device)
    y = y.to(device)

    return x, y


xb, yb = get_batch("train")

print("x shape:", xb.shape)
print("y shape:", yb.shape)

print()
print("Example x:")
print(decode(xb[0].tolist()))

print()
print("Example y:")
print(decode(yb[0].tolist()))

7. Attention Head

In [ ]:
class Head(nn.Module):
    """
    Single masked self-attention head
    """

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1)
        wei = wei * (k.shape[-1] ** -0.5)

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v

        return out

8. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multiple attention heads in parallel
    """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([
            head(x)
            for head in self.heads
        ], dim=-1)

        out = self.proj(out)
        out = self.dropout(out)

        return out

9. FeedForward Network

In [ ]:
class FeedForward(nn.Module):
    """
    Simple MLP applied to each position
    """

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

10. Transfomer Block

In [ ]:
class Block(nn.Module):
    """
    Transformer block:
    communication followed by computation
    """

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x

11. Small GPT 전체 모델

In [ ]:
class TinyGPT(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head)
            for _ in range(n_layer)
        ])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)

        pos = torch.arange(T, device=device)
        pos_emb = self.position_embedding_table(pos)

        x = tok_emb + pos_emb

        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, V = logits.shape

            logits_flat = logits.view(B * T, V)
            targets_flat = targets.view(B * T)

            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]
            logits = logits / temperature

            if top_k is not None:
                values, indices = torch.topk(logits, top_k)
                logits_filtered = torch.full_like(logits, float("-inf"))
                logits_filtered.scatter_(1, indices, values)
                logits = logits_filtered

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

12. 모델 생성

In [ ]:
model = TinyGPT().to(device)

num_params = sum(p.numel() for p in model.parameters())

print(model)
print()
print("Number of parameters:", num_params)
print("Number of parameters in millions:", num_params / 1e6)

13. Loss

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}

    model.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean().item()

    model.train()

    return out

14. 학습

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(
            f"step {iter}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

15. 텍스트 생성

In [ ]:
def safe_encode(s):
    return [stoi[ch] for ch in s if ch in stoi]


prompt = "Once upon a time"

start_ids = safe_encode(prompt)

if len(start_ids) == 0:
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
else:
    context = torch.tensor([start_ids], dtype=torch.long, device=device)

generated = model.generate(
    context,
    max_new_tokens=1000,
    temperature=0.9,
    top_k=30
)

result = decode(generated[0].tolist())

print(result)

16. 생성 결과 저장

In [ ]:
output_filename = "generated_text.txt"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(result)

print("Saved:", output_filename)

try:
    from google.colab import files
    files.download(output_filename)
except:
    pass

17. 모델 저장

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "stoi": stoi,
    "itos": itos,
    "config": {
        "vocab_size": vocab_size,
        "block_size": block_size,
        "batch_size": batch_size,
        "n_embd": n_embd,
        "n_head": n_head,
        "n_layer": n_layer,
        "dropout": dropout,
    }
}

model_filename = "tinygpt_char_model.pt"

torch.save(checkpoint, model_filename)

print("Model saved:", model_filename)

try:
    from google.colab import files
    files.download(model_filename)
except:
    pass

18. 마무리 설명

This project is a character-level TinyGPT text generator.

The model learns from an English text file and predicts the next character
based on the previous context. It uses token embeddings, positional embeddings,
masked multi-head self-attention, feedforward layers, residual connections,
layer normalization, and Transformer blocks.

After training, the model can generate new text that follows a similar style
to the training data.